In [1]:
"""
gridMET fire-weather climatology by Pyrome, over the MODIS fire-environment mask.

Exports daily GridMET variables (multi-percentile) per pyrome, restricted to the
same MODIS burned-area fire mask used by 00b_RTMA-EMC, across a configurable year
range and fire season. Runs on ANY pyrome subset — set REGION + PYROME_IDS in the
config cell (mirrors 00b). Output feeds fb_tools.weather.gridmet.load_gridmet_csv().

author: maxwell.cook@colostate.edu
"""

import ee
import geemap
import time

ee.Authenticate()
ee.Initialize(project='cfri-ee')
print("GEE Authenticated !")

GEE Authenticated !


In [2]:
# ── Analysis region config ───────────────────────────────────────────────
# Mirrors 00b_RTMA-EMC.ipynb. Edit REGION + PYROME_IDS for any pyrome subset;
# START_YEAR/END_YEAR set the climatology window (inclusive).
#   CO        : [42, 43, 45, 46, 47, 52, 53, 56, 128]
#   R4        : [14, 16, 44]                                   # WUC (44) + Payette (14, 16)
#   InterWest : [42,43,45,46,47,52,53,56,128, 14,16,44]        # CO + R4 combined
REGION     = 'InterWest'
PYROME_IDS = [42, 43, 45, 46, 47, 52, 53, 56, 128, 14, 16, 44]
START_YEAR = 2010
END_YEAR   = 2025

# --- Load Pyromes and filter to the selected IDs
pyromes = ee.FeatureCollection('projects/cfri-ee/assets/weather/Pyromes_CONUS_20200206')
region_pyromes = pyromes.filter(ee.Filter.inList('PYROME', PYROME_IDS))
print(f'{REGION} pyromes:', region_pyromes.size().getInfo())
region_pyromes.aggregate_array('PYROME').getInfo()

InterWest pyromes: 12


[14, 16, 42, 43, 44, 45, 46, 47, 52, 53, 56, 128]

In [3]:
# --- Load the GridMET ImageCollection
gridmet = ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
print(f"gridMET bands:\n\n{gridmet.first().bandNames().getInfo()}")

gridMET bands:

['pr', 'rmax', 'rmin', 'sph', 'srad', 'th', 'tmmn', 'tmmx', 'vs', 'erc', 'eto', 'bi', 'fm100', 'fm1000', 'etr', 'vpd']


In [4]:
# ── Fire-environment mask (MODIS burned area) ────────────────────────────
# Use the SAME materialized MODIS burned-area mask that 00b_RTMA-EMC built for
# this REGION, so the gridMET ERC climatology and the RTMA fuel moisture are
# sampled over IDENTICAL fire-environment pixels. This replaces the older FOD
# 5-km-buffer mask (kept in git history): MODIS burned area is observed fire
# extent rather than an ignition-point proxy, needs no FOD asset/spatial join,
# and keeps ERC (00a) consistent with FM (00b) for the paired scenario builder.
#
# Prereq: the asset must already exist — it does if you've run the 00b fire-mask
# export cell for this REGION (the RTMA monthly exports depend on it).
#
# Resolution note: the mask is 2500 m (EPSG:4326); gridMET is ~4638 m. The mask
# is FINER than gridMET, so updateMask + reduceRegions restricts gridMET cells
# to the fire environment cleanly (EE reprojects the 0/1 mask to the analysis
# grid). Cell-boundary accounting is approximate but second-order.
FIRE_MASK_ASSET = f'projects/cfri-ee/assets/weather/{REGION.lower()}_fire_mask_modis_2500m'
fire_mask = ee.Image(FIRE_MASK_ASSET).selfMask()
print('Loaded fire mask asset:', FIRE_MASK_ASSET)

# Sanity: fire-environment coverage per pyrome (mean of 0/1 = fraction burned).
# A pyrome with very low coverage will have few gridMET cells feeding its ERC.
cov = (
    fire_mask.unmask(0)
      .reduceRegions(collection=region_pyromes, reducer=ee.Reducer.mean(),
                     scale=2500, tileScale=4)
      .getInfo()
)
for f in sorted(cov['features'], key=lambda x: x['properties'].get('PYROME', 0)):
    pid  = f['properties'].get('PYROME')
    frac = f['properties'].get('mean', 0) or 0
    flag = ' ⚠ low' if frac < 0.05 else ''
    print(f"  Pyrome {pid:>4}: {frac*100:5.1f}% fire-environment pixels{flag}")

Loaded fire mask asset: projects/cfri-ee/assets/weather/interwest_fire_mask_modis_2500m
  Pyrome   14:  57.4% fire-environment pixels
  Pyrome   16:  38.2% fire-environment pixels
  Pyrome   42:   4.9% fire-environment pixels ⚠ low
  Pyrome   43:  17.5% fire-environment pixels
  Pyrome   44:  29.6% fire-environment pixels
  Pyrome   45:  18.2% fire-environment pixels
  Pyrome   46:  20.4% fire-environment pixels
  Pyrome   47:  16.4% fire-environment pixels
  Pyrome   52:   6.7% fire-environment pixels
  Pyrome   53:  39.7% fire-environment pixels
  Pyrome   56:  12.6% fire-environment pixels
  Pyrome  128:   5.7% fire-environment pixels


In [5]:
"""
Filter GridMET to the fire season over [START_YEAR, END_YEAR], mask each daily
image to the MODIS fire environment, and reduce per pyrome with a multi-
percentile reducer.

Per-day reduction: for each daily image, updateMask(fire_mask) then compute the
[10, 25, 50, 75, 90] percentile of each variable across the fire-environment
gridMET cells within each pyrome.  Output columns carry a `_pXX` suffix (e.g.,
`erc_p75`).  Downstream, `load_gridmet_csv(tail_percentile=...)` selects the
fire-dangerous tail at analysis time — e.g., p75 of hot/dry variables (tmmx,
erc, vs, vpd) with p25 of moisture variables (fm100, rmin, rmax) ≈ the driest
quartile of fire-environment cells.
"""

# --- Select bands
gridmet = gridmet.select(['vpd','erc','fm100','fm1000','tmmx','tmmn','rmax','rmin','pr','vs','th'])

gridmet_fs = gridmet.filter(ee.Filter.And(
    ee.Filter.calendarRange(START_YEAR, END_YEAR, 'year'),
    ee.Filter.calendarRange(4, 10, 'month')
)).filterBounds(region_pyromes)

n_images = gridmet_fs.size().getInfo()
n_pyromes = region_pyromes.size().getInfo()
n_years = END_YEAR - START_YEAR + 1
print(f"GridMET fire-season images: {n_images}  (expect ~{n_years}yrs × 214days = {n_years*214})")
print(f"Pyromes in analysis area:   {n_pyromes}")
print(f"Expected output features:   ~{n_images * n_pyromes:,}")

# Multi-percentile reducer: returns `<band>_p10`, `<band>_p25`, `<band>_p50`,
# `<band>_p75`, `<band>_p90` per variable per feature.
percentile_reducer = ee.Reducer.percentile([10, 25, 50, 75, 90])

image_list = gridmet_fs.toList(gridmet_fs.size())

def extract_day(img):
    img = ee.Image(img)
    date_str = img.date().format('YYYY-MM-dd')
    year = img.date().get('year')
    doy = img.date().getRelative('day', 'year').add(1)  # 1-indexed DOY

    sampled = (
        img.updateMask(fire_mask)          # restrict to MODIS fire-environment pixels
           .reduceRegions(
               collection=region_pyromes,   # full pyrome polygons; mask selects cells
               reducer=percentile_reducer,
               scale=4000,
           )
           .map(lambda f: f.set({
               'PYROME': f.get('PYROME'),
               'date': date_str,
               'year': year,
               'doy': doy,
           }))
    )
    return sampled

all_samples_fc = ee.FeatureCollection(image_list.map(extract_day)).flatten()

# Retain meta columns + all percentile-suffix variable columns.
BANDS = ['vpd','erc','fm100','fm1000','tmmx','tmmn','rmax','rmin','pr','vs','th']
PCTLS = [10, 25, 50, 75, 90, 97]
keep_cols = ['PYROME', 'date', 'year', 'doy'] + [f'{b}_p{p}' for b in BANDS for p in PCTLS]
all_samples_fc = all_samples_fc.select(keep_cols)
print(f"FeatureCollection ready. {len(keep_cols)} columns "
      f"({len(BANDS)} bands × {len(PCTLS)} percentiles + 4 meta).")
print("  NOTE: tmmx/tmmn in °K — fb_tools.weather.gridmet.load_gridmet_csv() converts to °F.")

GridMET fire-season images: 3424  (expect ~16yrs × 214days = 3424)
Pyromes in analysis area:   12
Expected output features:   ~41,088
FeatureCollection ready. 70 columns (11 bands × 6 percentiles + 4 meta).
  NOTE: tmmx/tmmn in °K — fb_tools.weather.gridmet.load_gridmet_csv() converts to °F.


In [6]:
# --- Export to Google Drive

def drop_geometry(feature):
    return feature.setGeometry(None)

all_samples_fc = all_samples_fc.map(drop_geometry)

# Name derives from REGION so CO and R4 exports never collide. The
# `_fmask_pctiles` suffix distinguishes this percentile-suffix export from the
# legacy pyrome-median CSV.
EXPORT_NAME = f'GRIDMET_ERC_{REGION}_pyromes'

export_task = ee.batch.Export.table.toDrive(
    collection=all_samples_fc,
    description=EXPORT_NAME,
    folder='fb_tools_weather',
    fileNamePrefix=EXPORT_NAME,
    fileFormat='CSV'
)

export_task.start()
print(f"Export to Google Drive started: {EXPORT_NAME}.csv")
print(f"  Load with: load_gridmet_csv('{EXPORT_NAME}.csv', tail_percentile=75)")

Export to Google Drive started: GRIDMET_ERC_InterWest_pyromes.csv
  Load with: load_gridmet_csv('GRIDMET_ERC_InterWest_pyromes.csv', tail_percentile=75)
